# KL Divergence for Model Quantization - Complete Tutorial

This notebook provides a comprehensive introduction to using KL divergence for evaluating model quantization quality, following the methodology used by Unsloth.

## Table of Contents
1. [Introduction to KL Divergence](#1.-Introduction-to-KL-Divergence)
2. [Why KL Divergence for Quantization?](#2.-Why-KL-Divergence-for-Quantization?)
3. [Basic KL Divergence Calculations](#3.-Basic-KL-Divergence-Calculations)
4. [Simulating Model Quantization](#4.-Simulating-Model-Quantization)
5. [Running Benchmarks](#5.-Running-Benchmarks)
6. [Visualizing Results](#6.-Visualizing-Results)
7. [Understanding the Results](#7.-Understanding-the-Results)

In [ ]:
# Setup
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from kl_divergence import KLDivergence, calculate_kl_divergence
from quantization import QuantizationType, ModelQuantizer, WeightQuantizer, compare_quantization_types
from benchmarks import QuantizationBenchmark, print_benchmark_results
from visualization import BenchmarkVisualizer

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All imports successful!")

## 1. Introduction to KL Divergence

### Mathematical Definition

The Kullback-Leibler (KL) divergence measures how much one probability distribution $Q$ differs from a reference distribution $P$:

$$D_{KL}(P \parallel Q) = \sum_{x} P(x) \log\frac{P(x)}{Q(x)}$$

### Key Properties

1. **Non-negative**: $D_{KL}(P \parallel Q) \geq 0$
2. **Zero if and only if identical**: $D_{KL}(P \parallel Q) = 0 \iff P = Q$
3. **Not symmetric**: $D_{KL}(P \parallel Q) \neq D_{KL}(Q \parallel P)$

### Why Use KL Divergence?

According to the paper **"Accuracy is Not All You Need"** and Unsloth's research:

- **Detects answer "flips"**: Changes from correct to incorrect (or vice versa)
- **More reliable than perplexity**: Perplexity can mask quality issues
- **Measures true fidelity**: How close the quantized model is to the original

## 2. Why KL Divergence for Quantization?

When we quantize a model (reduce precision from FP16 to 4-bit, for example), we want to know:

1. **How much has the model changed?**
2. **Will it give the same answers?**
3. **Which quantization method is best?**

KL divergence helps answer all three questions by measuring the difference between the output probability distributions of the original and quantized models.

## 3. Basic KL Divergence Calculations

Let's start with simple examples to understand KL divergence.

In [ ]:
# Example 1: Identical distributions
print("Example 1: Identical Distributions")
print("=" * 50)

p1 = np.array([0.5, 0.3, 0.2])
q1 = np.array([0.5, 0.3, 0.2])

kl1 = KLDivergence.calculate(p1, q1)

print(f"P: {p1}")
print(f"Q: {q1}")
print(f"KL(P || Q) = {kl1:.10f}")
print(f"\n✓ KL divergence is ~0 because distributions are identical\n")

In [ ]:
# Example 2: Slightly different distributions
print("Example 2: Slightly Perturbed Distribution")
print("=" * 50)

p2 = np.array([0.5, 0.3, 0.2])
q2 = np.array([0.48, 0.32, 0.2])  # Small changes

kl2 = KLDivergence.calculate(p2, q2)

print(f"Original:  {p2}")
print(f"Quantized: {q2}")
print(f"KL(P || Q) = {kl2:.6f}")
print(f"\n✓ Small changes → Small KL divergence\n")

In [ ]:
# Example 3: Demonstrating asymmetry
print("Example 3: KL Divergence is NOT Symmetric")
print("=" * 50)

p3 = np.array([0.7, 0.2, 0.1])
q3 = np.array([0.3, 0.4, 0.3])

kl_pq = KLDivergence.calculate(p3, q3)
kl_qp = KLDivergence.calculate(q3, p3)

print(f"P: {p3}")
print(f"Q: {q3}")
print(f"\nKL(P || Q) = {kl_pq:.6f}")
print(f"KL(Q || P) = {kl_qp:.6f}")
print(f"\nDifference: {abs(kl_pq - kl_qp):.6f}")
print(f"\n⚠️  Order matters! For quantization, we use KL(Original || Quantized)\n")

## 4. Simulating Model Quantization

Now let's simulate what happens when we quantize a language model. We'll compare different quantization types.

In [ ]:
# Show available quantization types
print("Available Quantization Types:")
print("=" * 80)
compare_quantization_types()

In [ ]:
# Simulate model outputs
print("Simulating Language Model Outputs")
print("=" * 80)

vocab_size = 10000  # Smaller vocab for demo
num_outputs = 1000  # Number of token predictions

quantizer = ModelQuantizer(seed=42)

# Generate realistic logits (peaky distributions like real LLMs)
logits = np.random.randn(num_outputs, vocab_size)
for i in range(num_outputs):
    # Boost top-k tokens (simulates LLM behavior)
    top_indices = np.random.choice(vocab_size, 100, replace=False)
    logits[i, top_indices] += np.random.uniform(2, 5, 100)

# Convert to probabilities
original_probs = quantizer.logits_to_probs(logits)

print(f"Generated {num_outputs} output distributions")
print(f"Vocabulary size: {vocab_size}")
print(f"\nExample distribution (first 10 tokens):")
print(original_probs[0, :10])
print(f"Sum of probabilities: {original_probs[0].sum():.6f} (should be 1.0)")

In [ ]:
# Compare different quantization types
print("\nQuantization Impact Analysis")
print("=" * 80)

test_types = [
    QuantizationType.Q2_K,
    QuantizationType.Q4_K_M,
    QuantizationType.Q6_K,
    QuantizationType.Q8_0,
    QuantizationType.F16,  # Full precision (baseline)
]

results_dict = {}

print(f"{'Quant Type':<15} {'Bits':<8} {'Size':<12} {'Mean KLD':<15} {'99.9% KLD':<15}")
print("-" * 80)

for quant_type in test_types:
    # Simulate quantization effect
    quantized_probs = WeightQuantizer.add_quantization_noise(
        original_probs,
        quant_type,
        seed=42
    )
    
    # Calculate KL statistics
    stats = KLDivergence.calculate_statistics(original_probs, quantized_probs)
    
    # Get quantization info
    from quantization import QUANT_CONFIGS
    config = QUANT_CONFIGS[quant_type]
    
    size_ratio = quantizer.get_size_reduction(quant_type)
    
    results_dict[quant_type.value] = stats
    
    print(f"{quant_type.value:<15} {config.bits:<8.1f} {size_ratio:<12.1%} "
          f"{stats['mean_kld']:<15.6f} {stats['kld_99_9']:<15.6f}")

print("-" * 80)
print("\n💡 Key Insights:")
print("   • Lower bits = smaller model but higher KL divergence")
print("   • Q4_K_M offers good balance (28% size, low KLD)")
print("   • F16 (full precision) has near-zero KLD (baseline)")

## 5. Running Benchmarks

Now let's run a comprehensive benchmark suite, just like Unsloth does for their GGUF quantizations.

In [ ]:
# Initialize benchmark
benchmark = QuantizationBenchmark(
    vocab_size=10000,
    num_samples=500,  # Reduced for speed
    seed=42
)

# Run benchmark suite
print("Running Comprehensive Benchmark Suite...")
print("This simulates benchmarking a 35GB model (like Qwen3.6-35B)\n")

benchmark_results = benchmark.run_benchmark_suite(
    base_model_size_gb=35.0,
    show_progress=True
)

# Display results
print("\n" + "=" * 80)
print_benchmark_results(benchmark_results)

In [ ]:
# Save results
benchmark.save_results(benchmark_results, "../results/tutorial_benchmark.json")
print("✓ Results saved to ../results/tutorial_benchmark.json")

## 6. Visualizing Results

Let's create visualizations similar to Unsloth's benchmark graphs.

In [ ]:
# Initialize visualizer
viz = BenchmarkVisualizer(figsize=(14, 8))

# Plot 1: KL Divergence vs Model Size (Unsloth's signature plot)
viz.plot_kld_vs_size(
    benchmark_results,
    metric='mean_kld',
    title='GGUF Performance Benchmarks\nMean KL Divergence vs Model Size',
    show_annotations=True
)

In [ ]:
# Plot 2: 99.9% KL Divergence (for outlier detection)
viz.plot_kld_vs_size(
    benchmark_results,
    metric='kld_99_9',
    title='99.9% KL Divergence vs Model Size\n(Outlier Detection)',
    show_annotations=False
)

In [ ]:
# Plot 3: Pareto Frontier (optimal trade-offs)
viz.plot_pareto_frontier(
    benchmark_results,
    metric='mean_kld'
)

In [ ]:
# Plot 4: Bits per weight vs KL divergence
viz.plot_bits_vs_kld(benchmark_results)

In [ ]:
# Plot 5: Complete dashboard
viz.create_summary_dashboard(benchmark_results)

## 7. Understanding the Results

### What Do These Metrics Mean?

1. **Mean KLD**: Average divergence across all outputs
   - Primary metric for overall quality
   - Lower = better fidelity to original model

2. **99.9% KLD**: Captures outliers without being too sensitive
   - Unsloth's preferred metric for robustness
   - Helps identify quantizations that occasionally produce bad outputs

3. **Max KLD**: Worst-case scenario
   - Important for critical applications
   - Very sensitive to single bad predictions

### Unsloth's Key Findings

From their research:

- **Layer sensitivity varies**: Some layers (like `ssm_out`, `attn_*`) are very sensitive to quantization
- **iMatrix helps**: Importance matrix quantization reduces KLD
- **Dynamic quantization works**: Quantizing each layer differently (Dynamic 2.0) minimizes KLD
- **KLD ≠ everything**: Must also validate with real benchmarks (MMLU, etc.)

### Practical Recommendations

Based on KL divergence analysis:

- **For maximum quality**: Use Q6_K or Q8_0
- **For best balance**: Use Q4_K_M or Q4_K_XL
- **For maximum compression**: Use Q2_K_XL with Dynamic quantization
- **Always validate**: KLD is a guide, test on your actual use case

## Conclusion

You've learned:

✅ What KL divergence is and why it's important  
✅ How to calculate KL divergence from scratch  
✅ How to simulate model quantization  
✅ How to benchmark different quantization types  
✅ How to visualize results like Unsloth  
✅ How to interpret KL divergence metrics  

### Next Steps

1. Run the full benchmark suite with larger vocabulary
2. Compare different "providers" (simulate competing quantization methods)
3. Analyze layer-by-layer sensitivity
4. Validate with real model quantizations

### Resources

- [Unsloth Dynamic 2.0 Documentation](https://unsloth.ai/docs/basics/unsloth-dynamic-2.0-ggufs)
- ["Accuracy is Not All You Need" Paper](https://arxiv.org/pdf/2407.09141)
- [Unsloth Qwen3.5 Benchmarks](https://unsloth.ai/docs/models/qwen3.5/gguf-benchmarks)